# 03 — Event-Type Decomposition (RQ2)

**Question.** Is the events-vs-fatalities discrepancy driven by a shift in the *mix* of event types (more low-lethality categories such as Protests and Strategic developments) or by changes in *per-event lethality within* types (e.g., Battles becoming more or less deadly)?

**Method.**
1. Aggregate to `YEAR × EVENT_TYPE`; visualise 100%-stacked area charts of the event-type share and the fatality share.
2. Compute per-event lethality (`FATALITIES / EVENTS`) by type-year.
3. **Shift-share decomposition** of the change in aggregate lethality $L_t = F_t / E_t = \sum_k s_{k,t} \, l_{k,t}$, where $s_{k,t}$ is the share of events in type $k$ and $l_{k,t}$ is type-$k$ lethality. Decompose $\Delta L = L_1 - L_0$ into:
   - **Composition effect** $\sum_k (s_{k,1} - s_{k,0}) \, l_{k,0}$ — pure mix shift at base-period lethalities.
   - **Within-type effect** $\sum_k s_{k,0} \, (l_{k,1} - l_{k,0})$ — pure lethality change at base-period mix.
   - **Interaction** $\sum_k (s_{k,1} - s_{k,0})(l_{k,1} - l_{k,0})$.
   This is the standard Oaxaca-style decomposition and is appropriate because aggregate lethality is *exactly* a weighted average of type-specific lethalities — the decomposition is an identity, not a model approximation.
4. Repeat with `DISORDER_TYPE` as a coarser robustness check.

**Takeaway.** Reported at the bottom.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

PROJECT = Path('/Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5')
DERIVED = PROJECT / 'data' / 'derived'
FIGDIR = PROJECT / 'docs' / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(DERIVED / 'acled_clean.parquet')
print('Loaded:', df.shape)

Loaded: (838798, 15)


## Aggregate to `YEAR × EVENT_TYPE`

In [2]:
by_type = (df.groupby(['YEAR','EVENT_TYPE'])[['EVENTS','FATALITIES']].sum().reset_index())
by_type['LETHALITY'] = by_type['FATALITIES'] / by_type['EVENTS'].replace(0, np.nan)
year_tot = by_type.groupby('YEAR')[['EVENTS','FATALITIES']].sum().rename(columns={'EVENTS':'EVENTS_TOT','FATALITIES':'FATALITIES_TOT'})
by_type = by_type.merge(year_tot, on='YEAR')
by_type['EV_SHARE'] = by_type['EVENTS']/by_type['EVENTS_TOT']
by_type['FT_SHARE'] = by_type['FATALITIES']/by_type['FATALITIES_TOT']
by_type.head(12)

,YEAR,EVENT_TYPE,EVENTS,FATALITIES,LETHALITY,EVENTS_TOT,FATALITIES_TOT,EV_SHARE,FT_SHARE
0,2014,Battles,5911,25028,4.234140,24662,47069,0.239680,0.531730
1,2014,Explosions/Remote violence,2548,6572,2.579278,24662,47069,0.103317,0.139625
2,2014,Protests,7165,108,0.015073,24662,47069,0.290528,0.002295
3,2014,Riots,2909,1026,0.352699,24662,47069,0.117955,0.021798
4,2014,Strategic developments,1199,83,0.069224,24662,47069,0.048617,0.001763
5,2014,Violence against civilians,4930,14252,2.890872,24662,47069,0.199903,0.302790
6,2015,Battles,7802,32785,4.202128,36222,62546,0.215394,0.524174
7,2015,Explosions/Remote violence,8258,15660,1.896343,36222,62546,0.227983,0.250376
8,2015,Protests,9541,256,0.026832,36222,62546,0.263403,0.004093
9,2015,Riots,4393,1361,0.309811,36222,62546,0.121280,0.021760


## 100%-stacked area charts: event share vs. fatality share by type

In [3]:
TYPE_ORDER = ['Battles','Explosions/Remote violence','Violence against civilians',
              'Riots','Protests','Strategic developments']
colors = {
    'Battles':'#8B0000',
    'Explosions/Remote violence':'#D62728',
    'Violence against civilians':'#FF7F0E',
    'Riots':'#FFBB78',
    'Protests':'#1F77B4',
    'Strategic developments':'#7F7F7F',
}

fig_share = make_subplots(rows=1, cols=2, subplot_titles=('Share of EVENTS by type','Share of FATALITIES by type'))
for t in TYPE_ORDER:
    s = by_type[by_type['EVENT_TYPE']==t].sort_values('YEAR')
    fig_share.add_trace(go.Scatter(x=s['YEAR'], y=s['EV_SHARE'], name=t, legendgroup=t,
                                   stackgroup='ev', groupnorm='percent',
                                   line=dict(width=0.5, color=colors[t]),
                                   fillcolor=colors[t]), row=1, col=1)
    fig_share.add_trace(go.Scatter(x=s['YEAR'], y=s['FT_SHARE'], name=t, legendgroup=t, showlegend=False,
                                   stackgroup='ft', groupnorm='percent',
                                   line=dict(width=0.5, color=colors[t]),
                                   fillcolor=colors[t]), row=1, col=2)
fig_share.update_yaxes(title_text='% of events', range=[0,100], row=1, col=1)
fig_share.update_yaxes(title_text='% of fatalities', range=[0,100], row=1, col=2)
fig_share.update_xaxes(dtick=1)
fig_share.update_layout(title='Global composition by EVENT_TYPE, 2014–2025', height=500, hovermode='x unified')
fig_share.write_html(FIGDIR / '03_share_stacked.html', include_plotlyjs='cdn')
fig_share.show()

## Per-event lethality by type-year

In [4]:
fig_leth_t = go.Figure()
for t in TYPE_ORDER:
    s = by_type[by_type['EVENT_TYPE']==t].sort_values('YEAR')
    fig_leth_t.add_trace(go.Scatter(x=s['YEAR'], y=s['LETHALITY'], name=t,
                                    mode='lines+markers', line=dict(color=colors[t], width=2.5)))
fig_leth_t.update_layout(title='Per-event lethality (fatalities / event) by EVENT_TYPE, 2014–2025',
                         xaxis_title='Year', yaxis_title='Fatalities per event',
                         yaxis_type='log', height=480, xaxis=dict(dtick=1), hovermode='x unified')
fig_leth_t.write_html(FIGDIR / '03_lethality_by_type.html', include_plotlyjs='cdn')
fig_leth_t.show()

## Shift-share decomposition of aggregate lethality

Decomposing $\Delta L = L_{2025} - L_{2014}$ into composition, within-type, and interaction components.

In [5]:
def shift_share(df_long, year0, year1, group_col):
    """Decompose change in aggregate lethality from year0 to year1 by group_col.
    Returns (L0, L1, comp, within, interaction)."""
    d0 = df_long[df_long['YEAR']==year0].set_index(group_col)
    d1 = df_long[df_long['YEAR']==year1].set_index(group_col)
    groups = sorted(set(d0.index) | set(d1.index))
    d0 = d0.reindex(groups).fillna(0)
    d1 = d1.reindex(groups).fillna(0)
    E0, E1 = d0['EVENTS'].sum(), d1['EVENTS'].sum()
    s0 = d0['EVENTS']/E0
    s1 = d1['EVENTS']/E1
    l0 = np.where(d0['EVENTS']>0, d0['FATALITIES']/d0['EVENTS'].replace(0,np.nan), 0)
    l1 = np.where(d1['EVENTS']>0, d1['FATALITIES']/d1['EVENTS'].replace(0,np.nan), 0)
    L0 = float((s0*l0).sum()); L1 = float((s1*l1).sum())
    comp   = float(((s1 - s0) * l0).sum())
    within = float((s0 * (l1 - l0)).sum())
    inter  = float(((s1 - s0) * (l1 - l0)).sum())
    return L0, L1, comp, within, inter, pd.DataFrame({
        group_col: groups, 's0': s0.values, 's1': s1.values,
        'l0': l0, 'l1': l1,
        'd_comp': (s1-s0).values * l0,
        'd_within': s0.values * (l1-l0),
        'd_inter': (s1-s0).values * (l1-l0)
    })

L0, L1, comp, within, inter, contrib = shift_share(by_type, 2014, 2025, 'EVENT_TYPE')
dL = L1 - L0
print(f'L(2014) = {L0:.3f}')
print(f'L(2025) = {L1:.3f}')
print(f'ΔL      = {dL:+.3f}')
print('-'*40)
print(f'Composition effect   = {comp:+.3f}  ({comp/dL*100:+.1f}% of ΔL)')
print(f'Within-type effect   = {within:+.3f}  ({within/dL*100:+.1f}% of ΔL)')
print(f'Interaction          = {inter:+.3f}  ({inter/dL*100:+.1f}% of ΔL)')
print(f'Check (sum)          = {comp+within+inter:+.3f}')
contrib.round(4)

L(2014) = 1.909
L(2025) = 0.593
ΔL      = -1.316
----------------------------------------
Composition effect   = -0.361  (+27.4% of ΔL)
Within-type effect   = -1.080  (+82.1% of ΔL)
Interaction          = +0.126  (-9.5% of ΔL)
Check (sum)          = -1.316


,EVENT_TYPE,s0,s1,l0,l1,d_comp,d_within,d_inter
0,Battles,0.2397,0.1586,4.2341,2.1599,-0.3434,-0.4972,0.1682
1,Explosions/Remote violence,0.1033,0.2132,2.5793,0.5776,0.2833,-0.2068,-0.2199
2,Protests,0.2905,0.3776,0.0151,0.0010,0.0013,-0.0041,-0.0012
3,Riots,0.1180,0.0459,0.3527,0.1820,-0.0254,-0.0201,0.0123
4,Strategic developments,0.0486,0.1018,0.0692,0.0089,0.0037,-0.0029,-0.0032
5,Violence against civilians,0.1999,0.1029,2.8909,1.1444,-0.2804,-0.3491,0.1694


In [6]:
# Per-type contribution bar chart (composition vs within)
contrib_plot = contrib.set_index('EVENT_TYPE').loc[TYPE_ORDER]
fig_dec = go.Figure()
fig_dec.add_trace(go.Bar(x=contrib_plot.index, y=contrib_plot['d_comp'],
                         name='Composition (share shift × base lethality)', marker_color='steelblue'))
fig_dec.add_trace(go.Bar(x=contrib_plot.index, y=contrib_plot['d_within'],
                         name='Within-type (base share × lethality change)', marker_color='firebrick'))
fig_dec.add_trace(go.Bar(x=contrib_plot.index, y=contrib_plot['d_inter'],
                         name='Interaction', marker_color='grey'))
fig_dec.update_layout(title=f'Shift-share decomposition of ΔL (2014→2025): L went {L0:.2f}→{L1:.2f}',
                      barmode='relative', yaxis_title='Contribution to ΔL (fatalities/event)',
                      height=480, xaxis_tickangle=-20)
fig_dec.write_html(FIGDIR / '03_decomposition.html', include_plotlyjs='cdn')
fig_dec.show()

## Year-by-year rolling decomposition

Decompose each year's $\Delta L$ from the prior year to see when the compositional shift mattered most.

In [7]:
years = sorted(by_type['YEAR'].unique())
rows = []
for y0, y1 in zip(years[:-1], years[1:]):
    L0,L1,c,w,i,_ = shift_share(by_type, y0, y1, 'EVENT_TYPE')
    rows.append({'YEAR':y1, 'L0':L0, 'L1':L1, 'dL':L1-L0,
                 'comp':c, 'within':w, 'inter':i})
roll = pd.DataFrame(rows)
print(roll.round(3))

fig_roll = go.Figure()
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['comp'],   name='Composition', marker_color='steelblue'))
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['within'], name='Within-type', marker_color='firebrick'))
fig_roll.add_trace(go.Bar(x=roll['YEAR'], y=roll['inter'],  name='Interaction', marker_color='grey'))
fig_roll.add_trace(go.Scatter(x=roll['YEAR'], y=roll['dL'], name='ΔL (total)',
                              mode='lines+markers', line=dict(color='black', width=2)))
fig_roll.update_layout(title='Year-on-year shift-share decomposition of lethality change',
                       barmode='relative', xaxis_title='Year', yaxis_title='ΔL',
                       height=460, xaxis=dict(dtick=1))
fig_roll.write_html(FIGDIR / '03_rolling_decomp.html', include_plotlyjs='cdn')
fig_roll.show()

    YEAR     L0     L1     dL   comp  within  inter
0   2015  1.909  1.727 -0.182  0.023  -0.141 -0.064
1   2016  1.727  1.624 -0.103 -0.227   0.137 -0.013
2   2017  1.624  1.619 -0.005  0.538  -0.383 -0.160
3   2018  1.619  0.839 -0.780 -0.199  -0.629  0.048
4   2019  0.839  0.663 -0.176 -0.105  -0.081  0.010
5   2020  0.663  0.519 -0.144 -0.175   0.037 -0.006
6   2021  0.519  0.546  0.027 -0.051   0.091 -0.013
7   2022  0.546  0.528 -0.018  0.082  -0.077 -0.023
8   2023  0.528  0.568  0.040  0.042  -0.003  0.001
9   2024  0.568  0.637  0.069  0.034   0.039 -0.004
10  2025  0.637  0.593 -0.044  0.032  -0.068 -0.008


## Robustness: repeat with `DISORDER_TYPE`

In [8]:
by_dis = (df.groupby(['YEAR','DISORDER_TYPE'])[['EVENTS','FATALITIES']].sum().reset_index())
L0,L1,c,w,i,contrib_d = shift_share(by_dis, 2014, 2025, 'DISORDER_TYPE')
dL = L1-L0
print(f'DISORDER_TYPE decomposition 2014→2025')
print(f'L0={L0:.3f}  L1={L1:.3f}  ΔL={dL:+.3f}')
print(f'  Composition = {c:+.3f}  ({c/dL*100:+.1f}%)')
print(f'  Within      = {w:+.3f}  ({w/dL*100:+.1f}%)')
print(f'  Interaction = {i:+.3f}  ({i/dL*100:+.1f}%)')
contrib_d.round(4)

DISORDER_TYPE decomposition 2014→2025
L0=1.909  L1=0.593  ΔL=-1.316
  Composition = -0.280  (+21.3%)
  Within      = -1.209  (+91.9%)
  Interaction = +0.173  (-13.1%)


,DISORDER_TYPE,s0,s1,l0,l1,d_comp,d_within,d_inter
0,Demonstrations,0.3485,0.3916,0.0353,0.0033,0.0015,-0.0111,-0.0014
1,Political violence,0.5942,0.5059,3.1781,1.1671,-0.2807,-1.1950,0.1776
2,Political violence; Demonstrations,0.0086,0.0007,0.5070,0.5516,-0.0040,0.0004,-0.0004
3,Strategic developments,0.0486,0.1018,0.0692,0.0089,0.0037,-0.0029,-0.0032


## Takeaway

Numerical findings from this notebook feed the RQ2 section of the report; narrative interpretation deferred to the report draft.